<!-- DATA PROVIDER INSTRUCTIONS

1. Provide the name of your dataset, replacing the bracketed placeholder text.
2. Update the Registry of Open Data landing page URL, by replacing the bracketed placeholder text. The [REGISTRY_YAML_NAME] will correspond to the name of the YAML document in your pull request to the Registry of Open Data on Github, minus the .yaml file extension.
3. Remove these comment blocks when you have completed each section.

DATA PROVIDER INSTRUCTIONS -->

# Get to Know a Dataset: T2T Human Genomes Dataset

This notebook serves as a guided tour of the [T2T Genome Assembly Datasets](https://registry.opendata.aws/[REGISTRY_YAML_NAME]) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

<!-- DATA PROVIDER INSTRUCTIONS

The goal of this section is to orient users to the structure of your dataset. 

1. How are key prefixes and objects organized in your S3 bucket?
2. What kinds of filetypes are represented in your dataset?
3. Explain with text what users are expected to encounter, and then demonstrate with code the organizational framework you applied when creating your dataset.
4. The responses to each question section are meant to be expanded or replaced as dictated by your dataset

DATA PROVIDER INSTRUCTIONS -->

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

At the top level of our S3 bucket, we have a single key prefix "name" that in turn contains a README.md files describing all genome assemblies currently available. Each genome has a folder containing:

 1. A sample.yaml document descirbing the dataset
 2. At least 3 primary directories, which contain raw sequencing data, metadata, and genome assembly annotation data (labeled by genome version ex: v1.0)
    - raw_sequencing: all associated raw data from ont, pacbio hifi, and pore-c sequencing
    - metadata
    - v1.0: genome assembly fastas and all associated annotations, alignments, methylation, and other descriptive analyses. 




In [ ]:
s3://name/
├── README.txt
└── genomes/
    └── <accession_id>/ # one directory per sample/genome
        ├── sample.yaml # file descirbing all data contained within the directory
        ├── raw_sequencing/ #raw reads from sequencing runs (pods5, bam, fastq)
        │   |── ont/
        │   │   |── fastq/
        │   │   |── pod5/
        │   │   └── ubam/
        │   └── pacbio_hifi/
        │   │   |── fastq/   
        │   │   └── ubam/
        │   └── pore-c/
        │   │   |── fastq/   
        │   │   └── ubam/
        ├── metadata/
        └── v<major>.<minor>/ # one directory for each version of the assembly containing assembly fasta and QC stats
            ├── assembly/
            ├── annotation/
            ├── variants/
            ├── alignments/
            ├── methylation/
            └── fiberseq/ 
        

In [ ]:
# CODING GUIDELINES FOR DATA PROVIDER
#
# General notebook coding guidelines:
# 1. Assume that your reader understands the basics of Jupyter Notebooks, Python, and their Python environment.
#    The focus of this tutorial is on your dataset.
# 2. For library requirements, list the required libraries in a comment block in "requirements.txt" format
#    (https://pip.pypa.io/en/stable/reference/requirements-file-format/)
# 3. Demonstrate importing libraries with the assumption that the user has correctly installed the required
#    libraries.
# 4. List and load all library dependencies once, at this point of the notebook, unless a complicated dependency
#    set makes it unweildy.
# 5. Remember, the goal of this tutorial is a 101-level introduction to your dataset using common tools and libraries.
#    Examples using specialized environments and deep-diving methods are better suited to follow-up tutorials.
#
# CODING GUIDELINES FOR DATA PROVIDER

First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23
# polars >= 1.30.0
# matplotlib >= 3.10.3 

# Import the libraries required for this notebook
# Built-ins
import json
from pprint import pprint
# Installed libraries
import boto3, polars, matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

Next, we will define the location of our dataset, create our boto3 S3 client, and list the top level prefixes in our S3 bucket. Here we see there is only one top-level prefix in our bucket.

In [ ]:
# Location of the S3 bucket for this dataset
bucket = "name"

# List the top level of the bucket using boto3. Because this is a public bucket, we don't need to sign requests.
# Here we set the signature version to unsigned, which is required for public buckets.
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# Print the items in the top-level prefixes
for item in s3.list_objects_v2(Bucket=bucket, Delimiter='/')['CommonPrefixes']:
    print(item['Prefix'])

Looking into the top-level S3 prefix of our dataset, genomes, we see that each sample genome is separated into its own folder.

In [ ]:
# List the key prefixes within the top level 'almrrc2021' prefix
for item in s3.list_objects_v2(Bucket=bucket, Prefix='genomes/', Delimiter='/', MaxKeys=10)['CommonPrefixes']:
    print(item['Prefix'])

To look deeper at all raw seqeuncing data files located within a single genome accession perform the following command to see where specific data resides:

In [ ]:
# List the keys within the 'almrrc2021/almrrc2021-data-training' prefix.
for item in s3.list_objects_v2(Bucket=bucket, Prefix='genomes/<accession_id>/sequencing_data/', MaxKeys=100)['Contents']:
    print(item['Key'])

<!-- DATA PROVIDER INSTRUCTIONS
This section is meant to orient users of your dataset to the formats present in your dataset, particularly if your dataset includes formats that may be unfamiliar to a general data scientist audience. This section should include:

1. Explanation of data format(s) (very common formats can be very briefly described, while less common
   or domain specific formats should include more explanation as well as links to official documentation)
2. Explanation of why the data format was chosen for your dataset
3. Recommendations around software and tooling to work with this data format
4. Explanation of any dataset-specific aspects to your usage of the format
5. Description of AWS services that may be useful to users working with your data
DATA PROVIDER INSTRUCTIONS -->

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

These datasets span contain both raw seqeuncing data, genome assemblies, and genome annotations, in several standard bioinformatics formats:

- **POD5** - raw Oxford Nanopore signal data, wihtout basecalling. Dorado or another basecaller may be used to determine the nucleotide sequences from this data. 
- **BAM** - Unaligned, raw seqeuncing data from both PacBio HiFi and ONT ultra-long seqeuncing will come as a bam file. Additional aligned bame files may be available as a componanion to annotaitons, with corresponding index files (.bai). 
- **bgzip-compressed FASTQ** - Raw seqeuncing data with quality and methylation information for PacBio HiFI and ONT sequencing. 
- **bgzip-compressed FASTA** (.fa.gz + .fai index) - genome assemblies will be provided as both combined and seprate haplotype resolved fasta files for diploid genomes. 
- **bgzip-compressed GFF3** (.gff3.gz + .tbi index) - gene annotation.
- **bgzip-compressed VCF** (.vcf.gz + .tbi index) - variant calls.

<!-- DATA PROVIDER INSTRUCTIONS
The goal of this section is to demonstrate loading a portion of data from your dataset, and reveal something about its structure.
1. Load an object from S3
2. Show the structure of data in the object
DATA PROVIDER INSTRUCTIONS -->

### Q: Can you show us an example of downloading and loading data from your dataset?

As an example, let us load up and look into some package data as found in s3://name/README.txt.

In [ ]:


# First we'll load the data into a Python dictionary using the built-in json library
#file is placeholder for now

file_key = "s3://name/README.txt"

with s3.get_object(Bucket=bucket, Key=file_key)['Body'] as file_object:
    package_data = json.load(file_object)


EXAMPLE - REPLACE

First, let's take a look at the keys in our newly-loaded dataset. Here we see that top-level keys correspond to route IDs.

EXAMPLE - REPLACE

In [ ]:
### EXAMPLE - REPLACE

# pretty print a truncated list of keys in our dictionary
pprint(list(package_data.keys())[:10])

### EXAMPLE - REPLACE

EXAMPLE - REPLACE

Next we'll look at packages associated with the first route ID in our dataset to get a sense for the structure of this file. Here we note that each package ID in this file has dimensions as well as a planned service time duration.

EXAMPLE - REPLACE

In [ ]:
### EXAMPLE - REPLACE

# pretty print the structure an individual route record
pprint(package_data["RouteID_15baae2d-bf07-4967-956a-173d4036613f"])

### EXAMPLE - REPLACE

<!-- DATA PROVIDER INSTRUCTIONS
The goal here is to visualize some aspect of your dataset in order to help users understand it. In addition to helping users of your dataset understand the dataset, an additional goal is to impress!

Please demonstrate any data preprocessing or reshaping required for your visualization(s).

https://www.reddit.com/r/dataisbeautiful/ for inspiration.
DATA PROVIDER INSTRUCTIONS -->

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

We can generate plots showing the structure of different human centromere sequences, along with their methylation and other epigenetic marks using cenmap (https://github.com/logsdon-lab/CenMAP), a tool generated by our lab. This data and these tracks can also be modified to include additional annotation and epigenetic information. 

![Here's an example from the DLD1 cell line's chromosome 1 centromeres](tutorials/DLD1_DiMeLo-Seq/pngs/dld1_v1.0_chr1_DLD1_chr1_haplotype1-0000002:120171375-128645169.png)




Now we can generate these centromere plots for our newly generated DLD1 genome using the raw PacBio Hifi and ONT data.

In [ ]:
### EXAMPLE - REPLACE with links to actual data

conda install bioconda::cenmap

cenmap -i asm*.fa.gz -s DLD1 --hifi hifi*.fq.gz --ont ont*.bam

<!-- DATA PROVIDER INSTRUCTIONS
This section is less prescriptive / freeform than previous sections. The goal here is to show an opinionated example of answering a question using your data. The scale of your dataset may preclude a full example, and so feel free to limit the scope of this example (e.g. work on a subset of data). Users should be able to replicate your example in this notebook, and get a sense of how they would scale up.

A "toy" example is better than no example.

Ideally, your example would:
1. Transmit some of your domain & dataset experience to the reader, drawing on your own work as much as possible
2. Provide a jumping off point for users to extend your work, and do novel work of their own.

DATA PROVIDER INSTRUCTIONS -->

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

Using this dataset, we determined the precise location of the hypomethylated region called the “centromere dip region” (or CDR), which marks the site of the kinetochore. We determined the location of the CDR in each centromere using a tool we built in collaboration with the Eichler lab called CDR-Finder, which takes a diploid genome assembly (.fa), aligned ONT data from the same source genome (.bam), and coordinates of the centromeric regions (.bed) as input and outputs a BED file with the predicted locations of the CDRs. These locations are then manually reviewed and curated to ensure they are accurate. When possible, we also test if these CDRs are enriched with CENP-A chromatin by performing CENP-A DiMeLo-seq or long-read CUT&RUN. Since CENP-A chromatin provides the foundation for the assembly of the kinetochore during mitosis and meiosis, this additional experiment confirms that the CDRs are the actual site of the kinetochore.

<!-- DATA PROVIDER INSTRUCTIONS
This section is, like the previous one, intended to be freeform / non-prescriptive. The goal here is to provide a challenge to the community to do something novel with your dataset. That can either be novel in terms of the task, or novel in terms of methodological or computational approach.

Another way to consider this section, is as a wishlist. If you were less constrained by time, cost, skill, etc., what would you like to see achieved using these data? 

The challenge should, however, be somewhat realistic. A challenge that assumes e.g. original data collection, is likely to go unanswered.
DATA PROVIDER INSTRUCTIONS -->

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

This data could be used to determine how different regions of the genome (e.g., centromeres, segmental duplications, and other repeat-rich regions) vary in their genetic and epigenetic landscapes among humans. By combining this dataset with publicly available genome assemblies and associated data from population-scale efforts, like those of the Human Pangenome Reference Consortium (HPRC), Human Genome Structural Variation Consortium (HGSVC), and Asian PanGenome (APG), among others, one could determine the extent of natural variation among these individuals, which can serve as a baseline for understanding how these regions vary in disease.

# DATA PROVIDER: PLEASE REMEMBER TO CLEAR ALL OUTPUTS BEFORE COMMITTING TO YOUR GITHUB REPOSITORY